# Импорты

In [40]:
import re
import requests
from collections import Counter
from string import punctuation

import wikipediaapi
import pandas as pd
import spacy
from nltk.corpus import stopwords
from natasha import (
    Segmenter,
    MorphVocab,
    NewsEmbedding,
    NewsMorphTagger,
    NewsNERTagger,
    Doc,
)

# Загрузка текста

In [41]:
with open(r'C:\Users\ivanr\Documents\cod\kursach\texts\genp_text.txt', 'r', encoding='utf-8') as f:
    raw_text = f.read()

# Предобработка

In [42]:
STOPWORDS_RU = set(stopwords.words('russian'))
PUNCT_EXTRA = {'—', '«', '»', "''", '…', '–'}
NOISE = STOPWORDS_RU | set(punctuation) | PUNCT_EXTRA


def clean_raw_text(text: str) -> str:
    parts = []
    for line in text.splitlines():
        line = re.sub(r'^"', '', line)
        line = re.sub(r'"\s*$', ' ', line)
        line = line.replace('—', '-').replace('–', '-')
        line = line.replace('«', '"').replace('»', '"')
        line = re.sub(r'\s+', ' ', line).strip()
        if line:
            parts.append(line)
    return ' '.join(parts)


text = clean_raw_text(raw_text)
print(f'Текст загружен: {len(text)} символов')

Текст загружен: 458221 символов


# Вспомогательные функции

In [43]:
def counter_to_info(counter):
    """Counter → dict {entity: {count: N}}"""
    return {ent: {'count': count} for ent, count in counter.items()}


def merge_raw_stores_max(store1, store2):
    """Объединяет два словаря сущностей, для совпадений берёт максимальный count."""
    merged = {}
    for ent in set(store1) | set(store2):
        info1 = store1.get(ent)
        info2 = store2.get(ent)
        if info1 and info2:
            merged[ent] = {'count': max(info1['count'], info2['count'])}
        else:
            merged[ent] = (info1 or info2).copy()
    return merged

# NER: Natasha

In [44]:
segmenter = Segmenter()
morph_vocab = MorphVocab()
emb = NewsEmbedding()
morph_tagger = NewsMorphTagger(emb)
ner_tagger = NewsNERTagger(emb)

doc_natasha = Doc(text)
doc_natasha.segment(segmenter)
doc_natasha.tag_morph(morph_tagger)
for token in doc_natasha.tokens:
    token.lemmatize(morph_vocab)
doc_natasha.tag_ner(ner_tagger)

persons_natasha = [
    ' '.join(t.lemma for t in span.tokens).lower()
    for span in doc_natasha.spans
    if span.type == 'PER'
]

info_natasha = counter_to_info(Counter(persons_natasha))
print(f'Natasha: {len(persons_natasha)} упоминаний, {len(info_natasha)} уникальных')

Natasha: 1856 упоминаний, 341 уникальных


# NER: spaCy

Если модель ещё не загружена:
```
python -m spacy download ru_core_news_lg
```

In [45]:
nlp = spacy.load('ru_core_news_lg')

doc_spacy = nlp(text)

persons_spacy = [
    ' '.join(t.lemma_ for t in ent).lower()
    for ent in doc_spacy.ents
    if ent.label_ == 'PER'
]

info_spacy = counter_to_info(Counter(persons_spacy))
print(f'spaCy: {len(persons_spacy)} упоминаний, {len(info_spacy)} уникальных')

spaCy: 2045 упоминаний, 358 уникальных


# Объединение результатов

In [46]:
merged_persons = merge_raw_stores_max(info_natasha, info_spacy)
print(f'После объединения: {len(merged_persons)} уникальных сущностей')

После объединения: 534 уникальных сущностей


# Обогащение через Wikipedia

In [ ]:
wiki = wikipediaapi.Wikipedia(
    language='ru',
    user_agent='pelevin-ner-project',
)

PERSON_MARKERS = [
    # биографические глаголы — самый надёжный признак живого человека
    'родился', 'родилась', 'умер', 'умерла', 'скончался', 'скончалась',
    # литература и публицистика
    'писатель', 'писательница', 'поэт', 'поэтесса', 'драматург',
    'переводчик', 'сценарист', 'критик', 'публицист', 'журналист', 'литератор',
    # театр и кино
    'актёр', 'актриса', 'режиссёр',
    # музыка
    'музыкант', 'певец', 'певица', 'композитор',
    # изобразительное искусство
    'художник', 'скульптор', 'архитектор',
    # наука
    'учёный', 'историк', 'философ', 'психолог', 'социолог', 'лингвист',
    # политика и военное дело
    'политик', 'деятель', 'революционер', 'военачальник', 'полководец',
    # предпринимательство и техника
    'предприниматель', 'изобретатель', 'инженер',
    # авиация и космос
    'лётчик', 'космонавт',
    # спорт
    'спортсмен', 'спортсменка',
    # религия и мифология
    'богослов', 'священник', 'монах', 'святой', 'апостол', 'пророк',
    'бог', 'божество',
]

ACTIVITIES = [
    'писатель', 'писательница', 'поэт', 'поэтесса', 'драматург',
    'переводчик', 'сценарист', 'критик', 'публицист', 'журналист', 'литератор',
    'актёр', 'актриса', 'режиссёр',
    'музыкант', 'певец', 'певица', 'композитор',
    'художник', 'скульптор', 'архитектор',
    'учёный', 'историк', 'философ', 'психолог', 'социолог', 'лингвист',
    'политик', 'деятель', 'революционер', 'военачальник', 'полководец',
    'предприниматель', 'изобретатель', 'инженер',
    'лётчик', 'космонавт',
    'спортсмен', 'спортсменка',
    'богослов', 'священник', 'монах', 'святой', 'апостол', 'пророк',
    'бог', 'божество',
]

# признаки страниц-дизамбигуаций и страниц-фамилий
_DISAMBIGUATION_SIGNS = [
    'фамилия',
    'многозначный',
    'неоднозначность',
    'может означать',
    'матронимическ',
    'патронимическ',
]


def is_disambiguation(page) -> bool:
    s = page.summary[:300].lower() if page.summary else ''
    return not s or any(sign in s for sign in _DISAMBIGUATION_SIGNS)


def find_wiki_page(ent: str):
    page = wiki.page(ent.title())
    if page.exists() and not is_disambiguation(page):
        return page

    try:
        r = requests.get(
            'https://ru.wikipedia.org/w/api.php',
            params={
                'action': 'query', 'list': 'search',
                'srsearch': ent, 'format': 'json', 'srlimit': 1,
            },
            timeout=10,
        )
        results = r.json().get('query', {}).get('search', [])
        if results:
            p = wiki.page(results[0]['title'])
            if p.exists():
                return p
    except Exception:
        pass

    return None


final_persons = {}

for ent, info in merged_persons.items():
    page = find_wiki_page(ent)

    if page is None:
        continue

    first_part = page.summary[:600].lower() if page.summary else ''

    if not any(marker in first_part for marker in PERSON_MARKERS):
        continue

    canonical_name = page.title
    entity_activities = [act for act in ACTIVITIES if act in first_part]
    is_deity = 'бог' in entity_activities or 'божество' in entity_activities

    birth_year = death_year = None
    if not is_deity:
        years = re.findall(r'(1[0-9]{3}|20[0-9]{2})', first_part)
        if years:
            birth_year = int(years[0])
        if len(years) >= 2:
            death_year = int(years[1])

    if canonical_name in final_persons:
        final_persons[canonical_name]['count'] += info['count']
    else:
        final_persons[canonical_name] = {
            'count': info['count'],
            'activities': entity_activities,
            'birth_year': birth_year,
            'death_year': death_year,
        }

# Итоговая таблица

In [48]:
df_persons = pd.DataFrame.from_dict(final_persons, orient='index')
df_persons.index.name = 'entity'
df_persons.reset_index(inplace=True)
df_persons

,entity,count,activities,birth_year,death_year
0,Родина,2,[],NaN,NaN
1,Аллах,5,"[пророк, бог, божество]",NaN,NaN
2,"Шиффер, Клаудия",1,[актриса],1970.0,NaN
3,Маркиз де Сад,1,"[писатель, историк, философ, политик]",1740.0,1814.0
4,Будда Шакьямуни,2,[],NaN,NaN
...,...,...,...,...,...
92,"Хаббард, Лафайет Рональд",1,"[писатель, критик]",1911.0,1986.0
93,"Гитлер, Адольф",1,[деятель],1889.0,1945.0
94,"Есенин, Сергей Александрович",1,"[писатель, поэт]",1895.0,1925.0
95,"Исикава, Такубоку",1,"[поэт, критик]",1886.0,1912.0


In [49]:
pd.set_option('display.max_rows', None)
df_persons

,entity,count,activities,birth_year,death_year
0,Родина,2,[],NaN,NaN
1,Аллах,5,"[пророк, бог, божество]",NaN,NaN
2,"Шиффер, Клаудия",1,[актриса],1970.0,NaN
3,Маркиз де Сад,1,"[писатель, историк, философ, политик]",1740.0,1814.0
4,Будда Шакьямуни,2,[],NaN,NaN
5,Кулак,1,[режиссёр],NaN,NaN
6,"Тютчев, Фёдор Иванович",2,"[поэт, переводчик, публицист]",1803.0,1873.0
7,"Троцкий, Лев Давидович",2,"[деятель, революционер]",1879.0,1940.0
8,"Березовский, Борис",1,"[политик, предприниматель]",1946.0,2013.0
9,"Кинг, Стивен",1,[писатель],1947.0,NaN
